In [1]:
%pip install -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
!{sys.executable} -m pip install tiktoken beautifulsoup4 langdetect pypdf rank_bm25

In [3]:
!pip install "qdrant-client>=1.8.0,<1.9.0"


In [4]:
"""
Krishi-Sakhi — Cross-lingual grounded RAG over rural-livelihood scheme PDFs.
Built on the ask-it starter. This notebook owns the ONE-PASS ingest and a
query-time baseline. See README.md as the single source of truth.

README §5  — one ingest pass, query-time gates
README §6  — critical technical corrections (hard constraints)
README §14 — build order
"""

import os
import re
import uuid
from pathlib import Path

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

# README §7A — structure/token-aware chunking deps (approximate token budgeting via tiktoken)
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# README §7A — HTML-to-text cleaning at ingest (loophole fix; also used for .html sample docs)
from bs4 import BeautifulSoup

# README §7D — per-chunk language tag threaded through the pipeline (see also §9 item 3)
from langdetect import detect as _lang_detect, DetectorFactory
DetectorFactory.seed = 0  # deterministic langdetect

# NOTE (open item): PDF text extraction is required for the notebook to function at all
# (starter TEXT_EXTENSIONS has no .pdf). Using pypdf — flagged for confirmation, see README cross-check.
from pypdf import PdfReader


/home/jovyan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from dotenv import load_dotenv
import os

# 1. Point directly to the file (one folder up)
# 2. Force Jupyter to overwrite any cached empty variables
load_dotenv("../.env", override=True) 

# Verify it loaded
api_key = os.environ.get("OPENAI_API_KEY") 
print(f"Key loaded: {str(api_key)[:5]}...")

Key loaded: sk-0s...


In [6]:
from pathlib import Path
from dotenv import load_dotenv

_repo_root = Path.cwd()
if not (_repo_root / "requirements.txt").is_file():
    _repo_root = _repo_root.parent
load_dotenv(_repo_root / ".env", override=True)

_PLACEHOLDER_MARKERS = ("<VAYU_", "<YOUR_", "<COLLECTION", "<", "***")


def _require_env(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise EnvironmentError(
            f"Missing required environment variable {name!r}. "
            "Copy .env.example to .env at the ask-it repo root and set a real value "
            "(see 04_starter_kit/README.md)."
        )
    if any(marker in value for marker in _PLACEHOLDER_MARKERS):
        raise EnvironmentError(
            f"Environment variable {name!r} looks like a placeholder. "
            "Update ask-it/.env with real credentials."
        )
    return value


# Hosted Qdrant + Vayu Model as a Service — required secrets (no inline defaults)
QDRANT_URL = _require_env("QDRANT_URL")
QDRANT_API_KEY = _require_env("QDRANT_API_KEY")
LLM_OPENAI_API_KEY = _require_env("OPENAI_API_KEY")
EMBEDDING_OPENAI_API_KEY = _require_env("EMBEDDING_OPENAI_API_KEY")
OPENAI_BASE_URL = _require_env("OPENAI_BASE_URL")

# Models — override via .env if needed
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "").strip() or "Qwen/Qwen3-Embedding-8B"
CHAT_MODEL = os.environ.get("CHAT_MODEL", "").strip() or "openai/gpt-oss-120b"
COLLECTION_NAME = os.environ.get("COLLECTION_NAME", "").strip() or "krishi_sakhi_rag"

# Local knowledge: all readable files under this folder (recursive)
DOCS_DIR = "docs/"
# README §7A / open item — .pdf added so PDFs are actually ingested (starter omitted it)
TEXT_EXTENSIONS = {".md", ".markdown", ".txt", ".html", ".htm", ".rst", ".csv", ".pdf"}
EMBED_BATCH_SIZE = 32
# Qdrant's HTTP API rejects a single request over 32 MiB (33_554_432 bytes).
# Batch upserts client-side to stay well under that regardless of embedding dim/corpus size.
UPSERT_BATCH_SIZE = 64

# ---- README §7A — structure- and token-aware chunking constants (auditable at a glance) ----
CHUNK_TOKENS = 450          # target ~400–500 tokens per chunk
CHUNK_OVERLAP_TOKENS = 60   # ~60-token overlap
# README §7A — separator hierarchy respecting structure + Devanagari danda (।) sentence boundary.
# Exact literal per README §2 rule 6 — do not alter.
CHUNK_SEPARATORS = ["\n## ", "\n### ", "\n\n", "\n", "। ", ". ", " ", ""]
# tiktoken cl100k_base is an APPROXIMATION of Qwen's tokenizer — used only for size budgeting
# (README §7A: acceptable, goal is approximate budgeting not exactness).
_TIKTOKEN_ENC = tiktoken.get_encoding("cl100k_base")

# ---- README §6 Correction 2 / §7B — Qwen3 query-instruction prefix (query path ONLY, English always) ----
# Exact literal per model card; documents get NO prefix. Do not pass prompt_name to MaaS (§6 Corr. 2).
QWEN_INSTRUCT = "Instruct: Given a question, retrieve passages that answer it\nQuery:"

# ---- README §7E — grounded-answer prompt config ----
CHAT_TEMPERATURE = 0.0        # README §6 Correction 4 — default 0, tested fallback 0.1
CHAT_TEMPERATURE_FALLBACK = 0.1
# Exact literal refusal string (README §7E) — the eval harness §11 greps for this verbatim.
REFUSAL_STRING = "I could not find this in the provided documents."

In [7]:
# Use OPENAI_BASE_URL from the config cell — never hardcode tenant URLs in source.
embedding_openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=EMBEDDING_OPENAI_API_KEY)
llm_openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=LLM_OPENAI_API_KEY)


In [8]:
# README §6 Correction 5 — the authoritative hosted handle is (re)assigned in Cell 6.
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)


In [9]:
# Force the exact model name required by your API key
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-8B"

# Fetch embedding dimensionality from the hosted MaaS model.
response = embedding_openai_client.embeddings.create(
    input=["Hello world!"],
    model=EMBEDDING_MODEL,
)

embedding = response.data[0].embedding
embedding_size = len(embedding)
print(f"Embedding size: {embedding_size}")

Embedding size: 4096


In [10]:
VECTOR_SIZE = embedding_size


def ensure_collection(qdrant_client, recreate: bool = False) -> None:
    """Create the Qdrant collection sized for the hosted embedding model."""
    existing = {c.name for c in qdrant_client.get_collections().collections}
    if COLLECTION_NAME in existing:
        if recreate:
            qdrant_client.delete_collection(COLLECTION_NAME)
        else:
            print(f"Collection {COLLECTION_NAME!r} already exists; skip create.")
            return
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    print(f"Created collection {COLLECTION_NAME!r} (dim={VECTOR_SIZE}).")


def hosted_instance(collection_name=COLLECTION_NAME, vector_size=VECTOR_SIZE):
    """Connect to the hosted Vayu Vector DB and ensure the collection exists."""
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)
    existing = {c.name for c in client.get_collections().collections}
    if collection_name not in existing:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        print(f"Created hosted collection {collection_name!r} (dim={vector_size}) at {QDRANT_URL}")
    else:
        print(f"Hosted collection {collection_name!r} already exists at {QDRANT_URL}")
    return client


# README §3 / §6 Correction 5 / §9 item 1 — PLUMBING FIX (priority zero):
# use hosted_instance(), a real COLLECTION_NAME, NOT local_db() / the <COLLECTION_NAME> placeholder.
qdrant = hosted_instance()

# README §7D — payload SCHEMA CHANGE means a full re-ingest. recreate=True wipes the old vectors
# so the collection is clean before the one-pass ingest (README §14: re-ingest ONCE after schema final).
RECREATE_COLLECTION = False
ensure_collection(qdrant, recreate=RECREATE_COLLECTION)



Hosted collection 'knowledge_base_rag' already exists at https://totayo-7a1a6-paas-7217-22277-private.cloudservices.tatacommunications.com
Collection 'knowledge_base_rag' already exists; skip create.


In [12]:
from pathlib import Path
from rank_bm25 import BM25Okapi  # README §7C — client-side BM25 (NOT Qdrant native sparse; §6 Corr. 1)


# ---------------------------------------------------------------------------
# README §7A — file loading, HTML/PDF-to-text cleaning
# ---------------------------------------------------------------------------
def _clean_html(raw: str) -> str:
    """README §7A loophole fix — strip HTML tags before chunking."""
    return BeautifulSoup(raw, "html.parser").get_text(separator="\n")


# ---- PDF read WITH page boundaries so citations can show a page number (README §7D / UI) ----
def _read_pdf_pages(path: Path) -> list[tuple[int, str]]:
    reader = PdfReader(str(path))
    return [(i + 1, (page.extract_text() or "")) for i, page in enumerate(reader.pages)]


def _read_file_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return "\n\n".join(t for _, t in _read_pdf_pages(path))
    raw = path.read_text(encoding="utf-8", errors="replace")
    if suffix in {".html", ".htm"}:
        return _clean_html(raw)
    return raw


def iter_doc_files(root: Path):
    if not root.is_dir():
        return
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in TEXT_EXTENSIONS:
            yield path


# ---------------------------------------------------------------------------
# README §7A — structure/token-aware chunking; README §7D — payload fields
# ---------------------------------------------------------------------------
def _tiktoken_len(text: str) -> int:
    return len(_TIKTOKEN_ENC.encode(text))


_splitter = RecursiveCharacterTextSplitter(
    separators=CHUNK_SEPARATORS,
    chunk_size=CHUNK_TOKENS,
    chunk_overlap=CHUNK_OVERLAP_TOKENS,
    length_function=_tiktoken_len,
    is_separator_regex=False,
)


# Match markdown ## / ### AND common circular headings ("4.2 Eligibility", "Section 3", Devanagari).
_HEADING_RE = re.compile(
    r"^(?:#{2,3}\s+(?P<md>.+)"
    r"|(?P<num>\d+(?:\.\d+)*\.?\s+[A-Z][^\n]{2,80})"
    r"|(?:Section|SECTION|खंड|धारा)\s+[^\n]{1,60})$",
    re.MULTILINE,
)


def _extract_headings(text: str) -> list[tuple[int, str]]:
    out = []
    for m in _HEADING_RE.finditer(text):
        title = (m.groupdict().get("md") or m.groupdict().get("num") or m.group(0)).strip()
        out.append((m.start(), title))
    return out


def _section_at(offset: int, headings: list[tuple[int, str]]) -> str:
    current = ""
    for h_off, title in headings:
        if h_off <= offset:
            current = title
        else:
            break
    return current


# ---- Script-based language detection (deterministic; replaces noisy langdetect on tables) ----
def _detect_lang(text: str) -> str:
    for ch in text:
        if "\u0900" <= ch <= "\u097F":
            return "hi"
    return "en"

_PII_PATTERNS = [
    ("AADHAAR", re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b")),
    ("PAN", re.compile(r"\b[A-Z]{5}[0-9]{4}[A-Z]\b")),
    ("PHONE", re.compile(r"\b(?:\+91[-\s]?)?[6-9]\d{9}\b")),
]


def redact_pii(text: str) -> str:
    if not text:
        return text
    redacted = text
    for label, pattern in _PII_PATTERNS:
        redacted = pattern.sub(f"[REDACTED-{label}]", redacted)
    return redacted

def chunk_file_text(text: str, source: str, page_spans: list[tuple[int, int]] | None = None) -> list[dict]:
    """README §7A/§7D — chunks with CORRECT offsets + section + page + lang."""
    text = text.strip("\n")
    if not text.strip():
        return []

    headings = _extract_headings(text)
    pieces = _splitter.split_text(text)
    out: list[dict] = []
    cursor = 0  # monotonic scan pointer — prevents re-finding earlier overlapping occurrences
    for piece in pieces:
        if not piece.strip():
            continue
        found = text.find(piece, cursor)
        if found == -1:
            found = text.find(piece)  # fallback; rare
        char_start = found if found != -1 else cursor
        char_end = char_start + len(piece)
        # advance cursor past the START of this piece (not the end) so overlap still matches next time
        cursor = char_start + 1

        page = None
        if page_spans:
            for pg_no, span_start in page_spans:
                if span_start <= char_start:
                    page = pg_no
                else:
                    break

        out.append(
            {
                "text": piece,
                "source": source,
                "chunk_index": len(out),
                "char_start": char_start,
                "char_end": char_end,
                "section": _section_at(char_start, headings),
                "page": page,
                "lang": _detect_lang(piece),
            }
        )
    return out


def load_chunks_from_docs(docs_dir: Path) -> list[dict]:
    root = docs_dir
    out: list[dict] = []
    if not root.exists():
        root.mkdir(parents=True, exist_ok=True)
    if not root.is_dir():
        print(f"Docs path is not a folder: {root}")
        return out
    for file_path in iter_doc_files(root):
        try:
            page_spans = None
            if file_path.suffix.lower() == ".pdf":
                pages = _read_pdf_pages(file_path)
                raw_parts, spans, pos = [], [], 0
                for pg_no, ptext in pages:
                    spans.append((pg_no, pos))
                    raw_parts.append(ptext)
                    pos += len(ptext) + 2  # +2 accounts for the "\n\n" join
                raw = "\n\n".join(raw_parts)
                if file_path.suffix.lower() in {".html", ".htm"}:
                    raw = _clean_html(raw)
            else:
                raw = _read_file_text(file_path)
                spans = None
        except Exception as e:
            print(f"Skip {file_path}: {type(e).__name__}: {e}")
            continue
        rel = str(file_path.relative_to(root))
        chunks = chunk_file_text(raw, rel, page_spans=spans)
        print(f"  {rel}: {len(chunks)} chunks")
        out.extend(chunks)
    return out


# ---------------------------------------------------------------------------
# README §7B / §6 Correction 2 — documents vs. query embedding paths are SEPARATE.
# ---------------------------------------------------------------------------
def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed DOCUMENTS — NO instruction prefix."""
    if not texts:
        return []
    all_vectors: list[list[float]] = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i : i + EMBED_BATCH_SIZE]
        try:
            res = embedding_openai_client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
        except Exception as e:
            raise RuntimeError(f"MaaS embedding call failed for batch starting {i}: {e}") from e
        all_vectors.extend(d.embedding for d in res.data)
    return all_vectors


def embed_query(question: str) -> list[float]:
    """README §6 Corr. 2 / §7B — QUERY path only: manual English instruction prefix."""
    return embed_texts([f"{QWEN_INSTRUCT}{question}"])[0]


# ---------------------------------------------------------------------------
# README §7D — index with extended payload; batched to respect Qdrant 32 MiB limit
# ---------------------------------------------------------------------------
def upsert_chunks(chunks: list[dict]) -> None:
    if not chunks:
        print("No chunks to index; add files under", DOCS_DIR)
        return
    redacted_texts = [redact_pii(c["text"]) for c in chunks]
    vectors = embed_texts(redacted_texts)
    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=v,
            payload={
                "text": t,
                "source": c["source"],
                "chunk_index": c["chunk_index"],
                "char_start": c["char_start"],
                "char_end": c["char_end"],
                "section": c["section"],
                "page": c.get("page"),
                "lang": c["lang"],
            },
        )
        for v, t, c in zip(vectors, redacted_texts, chunks)
    ]
    for i in range(0, len(points), UPSERT_BATCH_SIZE):
        qdrant.upsert(collection_name=COLLECTION_NAME, points=points[i : i + UPSERT_BATCH_SIZE])
    print(f"Indexed {len(points)} chunks from {DOCS_DIR} into {COLLECTION_NAME!r}.")
       


# ---------------------------------------------------------------------------
# README §7C — client-side hybrid retrieval: dense + BM25 + RRF
# ---------------------------------------------------------------------------
DENSE_TOP_K = 20
BM25_TOP_K = 20
RRF_FUSE_TOP_K = 12
RRF_K = 60

_BM25 = None
_BM25_PAYLOADS: list[dict] = []


def _bm25_tokenize(text: str) -> list[str]:
    """README §7C — whitespace split; known limitation for Devanagari."""
    return text.split()


def build_bm25_index() -> None:
    global _BM25, _BM25_PAYLOADS
    payloads: list[dict] = []
    next_offset = None
    while True:
        records, next_offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            with_payload=True,
            with_vectors=False,
            limit=256,
            offset=next_offset,
        )
        payloads.extend(r.payload for r in records if r.payload)
        if next_offset is None:
            break
    if not payloads:
        raise RuntimeError("BM25 build found an empty collection — ingest before building the index.")
    _BM25_PAYLOADS = payloads
    _BM25 = BM25Okapi([_bm25_tokenize(p.get("text", "")) for p in payloads])
    print(f"Built BM25 index over {len(payloads)} chunks.")


def _dense_search(question: str, limit: int) -> list[dict]:
    """Dense semantic search via Qdrant. Loud on failure (README §5)."""
    qvec = embed_query(question)
    hits = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=qvec,
        limit=limit,
        with_payload=True,
    )
    return [h.payload for h in hits if h.payload]


def _bm25_search(question: str, limit: int) -> list[dict]:
    if _BM25 is None:
        raise RuntimeError("BM25 index not built — call build_bm25_index() after ingest.")
    scores = _BM25.get_scores(_bm25_tokenize(question))
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:limit]
    return [_BM25_PAYLOADS[i] for i in ranked]


def _rrf_fuse(dense: list[dict], lexical: list[dict], top_k: int) -> list[dict]:
    """README §7C — RRF: score(doc) = Σ 1 / (60 + rank)."""
    fused: dict[tuple, float] = {}
    payload_by_key: dict[tuple, dict] = {}

    def _key(p: dict) -> tuple:
        return (p.get("source"), p.get("chunk_index"))

    for lane in (dense, lexical):
        for rank, payload in enumerate(lane):
            k = _key(payload)
            fused[k] = fused.get(k, 0.0) + 1.0 / (RRF_K + rank)
            payload_by_key[k] = payload

    ordered = sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
    return [payload_by_key[k] for k, _ in ordered]


def retrieve(question: str) -> list[dict]:
    """README §7C — hybrid retrieve. Errors propagate (no silent empty)."""
    dense = _dense_search(question, DENSE_TOP_K)
    lexical = _bm25_search(question, BM25_TOP_K)
    return _rrf_fuse(dense, lexical, RRF_FUSE_TOP_K)


def debug_retrieve(question: str) -> None:
    """Run this if you get an unexpected refusal — shows exactly what each lane returns."""
    print(f"QUESTION: {question!r}")
    dense = _dense_search(question, DENSE_TOP_K)
    print(f"  dense hits: {len(dense)}")
    if dense:
        print(f"    top dense source/section: {dense[0].get('source')} · {dense[0].get('section')}")
        print(f"    top dense text[:200]: {dense[0].get('text', '')[:200]!r}")
    lexical = _bm25_search(question, BM25_TOP_K)
    print(f"  bm25 hits:  {len(lexical)}")
    if lexical:
        print(f"    top bm25 text[:200]: {lexical[0].get('text', '')[:200]!r}")
    fused = _rrf_fuse(dense, lexical, RRF_FUSE_TOP_K)
    print(f"  fused: {len(fused)}")


# ---------------------------------------------------------------------------
# README §7E — grounded prompt, same-language, fixed refusal, temp fallback
# ---------------------------------------------------------------------------
ANSWER_TOP_K = 4


def rag_answer(question: str) -> str:
    hits = retrieve(question)[:ANSWER_TOP_K]
    if not hits:
        return REFUSAL_STRING

    blocks = []
    for i, h in enumerate(hits, start=1):
        src = h.get("source", "")
        section = h.get("section", "")
        body = h.get("text", "")
        tag = f"[{i}] {src}" + (f" · {section}" if section else "")
        blocks.append(f"{tag}\n{body}")
    context = "\n\n---\n\n".join(blocks)

    system = (
        "You are Krishi-Sakhi, an assistant for India's rural livelihood schemes. "
        "Answer using ONLY the provided context. "
        "Attach a bracketed source tag like [1] after every factual sentence. "
        "Answer in the SAME language the question was asked in. "
        f'If the context does not contain the answer, reply exactly: "{REFUSAL_STRING}"'
    )
    user = f"Context:\n{context}\n\nQuestion: {question}"
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]

    # README §6 Correction 4 — temp=0 with tested 0.1 fallback.
    try:
        completion = llm_openai_client.chat.completions.create(
            model=CHAT_MODEL, messages=messages, temperature=CHAT_TEMPERATURE,
        )
    except Exception as e_zero:
        print(f"temperature={CHAT_TEMPERATURE} rejected ({e_zero}); retrying at {CHAT_TEMPERATURE_FALLBACK}.")
        completion = llm_openai_client.chat.completions.create(
            model=CHAT_MODEL, messages=messages, temperature=CHAT_TEMPERATURE_FALLBACK,
        )
    return completion.choices[0].message.content or ""


# ---------------------------------------------------------------------------
# README §5 / §14 — ONE-PASS ingest, build BM25, then a baseline query.
# ---------------------------------------------------------------------------
DOCS_DIR = Path("../01_dataset/docs")

chunks = load_chunks_from_docs(DOCS_DIR)
upsert_chunks(chunks)
build_bm25_index()

# ---- Diagnose first (this is why you saw a refusal): ----
debug_retrieve("Namo Drone Didi max subsidy?")

demo_q = "What categories of crops are eligible for coverage under the Pradhan Mantri Fasal Bima Yojana (PMFBY) scheme?"
print("\nQ:", demo_q)
print("A:", rag_answer(demo_q))

  FINALSchemeGuidelinesAIF.pdf: 24 chunks
  FPO_Scheme_Guidelines_FINAL_English.pdf: 90 chunks
  Fasal Bima Yojana.pdf: 200 chunks
  KCC-RURAL BANKS.pdf: 26 chunks
  KCC-RURAL COOPERATIVE BANKS.pdf: 27 chunks
  KUSUM 2024.pdf: 50 chunks
  Kisan Credit Card.pdf: 24 chunks
  OG-PMKMY(Kisan Maan Dhan)_ENG.pdf: 29 chunks
  Operational_Guidelines_of_Namo_Drone_Didi_Scheme.pdf: 63 chunks
  R B I Master Circular on DAY _ NRLM 2024-25.pdf: 48 chunks
  RevisedPM-KISANOperationalGuidelines(English).pdf: 20 chunks
  html-example.html: 1 chunks
  markdown-example.md: 1 chunks
  pmfme_guidelines_english.pdf: 49 chunks
  text-example.txt: 1 chunks
Indexed 653 chunks from ../01_dataset/docs into 'knowledge_base_rag'.
Built BM25 index over 1959 chunks.
QUESTION: 'Namo Drone Didi max subsidy?'
  dense hits: 20
    top dense source/section: Operational_Guidelines_of_Namo_Drone_Didi_Scheme.pdf · 
    top dense text[:200]: "F. No.3-3(Drone)/2023- M&T (l&P) (1 37 552)\nGovernment of lndia\nMinistry of Agri

In [13]:
from pathlib import Path
for p in Path("../01_dataset/docs").rglob("*"):
    if p.is_file():
        print(p.name)


OG-PMKMY(Kisan Maan Dhan)_ENG.pdf
Operational_Guidelines_of_Namo_Drone_Didi_Scheme.pdf
markdown-example.md
R B I Master Circular on DAY _ NRLM 2024-25.pdf
text-example.txt
KUSUM 2024.pdf
RevisedPM-KISANOperationalGuidelines(English).pdf
KCC-RURAL BANKS.pdf
Kisan Credit Card.pdf
pmfme_guidelines_english.pdf
html-example.html
FINALSchemeGuidelinesAIF.pdf
FPO_Scheme_Guidelines_FINAL_English.pdf
Fasal Bima Yojana.pdf
KCC-RURAL COOPERATIVE BANKS.pdf
